# Notebook 02 — Exploratory Data Analysis

**Goal:** Understand distributions, temporal patterns, cross-year trends, and surface the initial bunching signal.

**Data sources used in this notebook:**
- `aggregate_full.parquet` — 4.4 M rows, station×hour×route×date level, **2022–2026/05** (primary for trends)
- `strategic/YYYY-MM.parquet` — full row-level data, **2024–2026/05** (for distributions & regression)

## Sections
1. Setup & Load
2. Distribution Analysis — travel time, dwell time, headway by line
3. Cross-Year Trends — 2022–2026 bunching & volume trends (NEW)
4. Temporal Patterns — hour of day, day of week
5. Initial Bunching Signal — prevalence and hotspots
6. Headway Gap → Dwell Time — crowding inference preview (H2)
7. Green Line B Spotlight — surface vs. underground deep dive (H3)
8. Seasonality — full 5-year monthly view
9. Save Enriched Sample for Downstream Notebooks

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'

ROUTE_LINE_MAP = {
    'Red': 'Red Line', 'Orange': 'Orange Line', 'Blue': 'Blue Line',
    'Green-B': 'Green Line B', 'Green-C': 'Green Line C',
    'Green-D': 'Green Line D', 'Green-E': 'Green Line E',
    'Mattapan': 'Mattapan Trolley',
}
LINE_COLORS = {
    'Red Line': '#DA291C', 'Orange Line': '#ED8B00', 'Blue Line': '#003DA5',
    'Green Line B': '#00843D', 'Green Line C': '#00843D',
    'Green Line D': '#00843D', 'Green Line E': '#00843D',
    'Mattapan Trolley': '#80276C',
}

# ── Primary: aggregate_full (station×hour×route×date, 4.4M rows, 2022-2026/05) ──
agg = pd.read_parquet(PROC_DIR / 'aggregate_full.parquet')
print(f'aggregate_full : {len(agg):,} rows × {agg.shape[1]} cols')
print(f'Date range     : {agg.service_date.min().date()} → {agg.service_date.max().date()}')
print(f'Years          : {sorted(agg["year"].unique())}')
print(f'Lines          : {sorted(agg["line"].unique())}')
print()

# ── Row-level: 2024-10 for distribution plots (Oct = high ridership, typical service) ──
month_sample = pd.read_parquet(PROC_DIR / 'strategic' / '2024-10.parquet')
month_sample['stop_dt']   = (pd.to_datetime(month_sample['stop_timestamp'], unit='s', utc=True)
                               .dt.tz_convert('America/New_York'))
month_sample['hour']      = month_sample['stop_dt'].dt.hour
month_sample['dow']       = month_sample['stop_dt'].dt.dayofweek
month_sample['is_weekend']= month_sample['dow'] >= 5
print(f'strategic 2024-10: {len(month_sample):,} rows  (Oct 2024, row-level)')

## 2. Distribution Analysis — Key Metrics by Line

In [ ]:
# Summary stats from aggregate_full (weighted means of per-station medians)
line_stats = (
    agg.groupby('line')
    .agg(
        total_events      = ('n_events',        'sum'),
        total_headway_obs = ('n_headway',        'sum'),
        bunching_events   = ('bunching_events',  'sum'),
        approx_med_dwell  = ('median_dwell',     'mean'),   # mean of station-medians
        approx_med_travel = ('median_travel',    'mean'),
        approx_med_headway= ('median_headway',   'mean'),
    )
    .assign(bunching_rate_pct=lambda d:
            (d['bunching_events'] / d['total_headway_obs'] * 100).round(2))
    .round(1)
    .sort_values('total_events', ascending=False)
)
print('Line summary — full 2022–2026/05 dataset')
line_stats

In [ ]:
# Box plots of dwell + travel time (from 2024-10 row-level data)
lines_ordered = (
    month_sample.groupby('line')['travel_time_seconds'].median()
    .sort_values().index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col, title in [
    (axes[0], 'dwell_time_seconds',  'Dwell Time by Line (sec)'),
    (axes[1], 'travel_time_seconds', 'Travel Time by Line (sec)'),
]:
    cap = month_sample[col].quantile(0.99)
    data_by_line = [
        month_sample.loc[(month_sample['line'] == l) & (month_sample[col] <= cap), col].dropna()
        for l in lines_ordered
    ]
    bp = ax.boxplot(data_by_line, vert=True, patch_artist=True, showfliers=False)
    for patch, line in zip(bp['boxes'], lines_ordered):
        patch.set_facecolor(LINE_COLORS.get(line, 'gray'))
        patch.set_alpha(0.7)
    ax.set_xticklabels(
        [l.replace(' Line', '').replace(' Trolley', '') for l in lines_ordered],
        rotation=30, ha='right'
    )
    ax.set_title(title)
    ax.set_ylabel('Seconds')

fig.suptitle('Distribution of Key Metrics by Line — October 2024', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Headway distribution by line (lines with branch headway data)
hw_lines = (
    month_sample.groupby('line')['headway_branch_seconds'].count()
    .pipe(lambda s: s[s > 100].index.tolist())
)

fig, ax = plt.subplots(figsize=(12, 4))
for line in hw_lines:
    data = month_sample.loc[month_sample['line'] == line, 'headway_branch_seconds'].dropna()
    cap  = data.quantile(0.99)
    ax.hist(data[data <= cap], bins=80, alpha=0.5,
            color=LINE_COLORS.get(line, 'gray'), label=line, density=True)

ax.axvline(120, color='red', ls='--', lw=1.5, label='Bunching threshold (2 min)')
ax.set_xlabel('Branch Headway (seconds)')
ax.set_ylabel('Density')
ax.set_title('Headway Distribution by Line — October 2024')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Cross-Year Trends — 2022–2026 (NEW)

Using `aggregate_full` — the first time we can see multi-year patterns with this dataset.

In [ ]:
# Year-over-year bunching rate by line
# True rate = total bunching events / total headway observations per year+line
yoy = (
    agg[agg['year'] < 2026]   # exclude 2026 (partial year, Jan-May only)
    .groupby(['year', 'line'])
    .agg(bunching_events=('bunching_events', 'sum'),
         n_headway=('n_headway', 'sum'))
    .assign(bunching_rate=lambda d: d['bunching_events'] / d['n_headway'] * 100)
    .reset_index()
)

# Only lines with headway data (Green branches + Red)
hw_lines_agg = (
    agg.groupby('line')['n_headway'].sum()
    .pipe(lambda s: s[s > 10000].index.tolist())
)

fig, ax = plt.subplots(figsize=(13, 5))
for line in hw_lines_agg:
    sub = yoy[yoy['line'] == line].sort_values('year')
    ax.plot(sub['year'], sub['bunching_rate'],
            color=LINE_COLORS.get(line, 'gray'), lw=2.2, marker='o', ms=6,
            label=line.replace(' Line', '').replace(' Trolley', ''))

ax.set_xlabel('Year')
ax.set_ylabel('Bunching Rate (%)')
ax.set_title('Year-over-Year Bunching Rate by Line — 2022–2025')
ax.set_xticks([2022, 2023, 2024, 2025])
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.show()

print('Bunching rate by year and line:')
print(yoy[yoy['line'].isin(hw_lines_agg)]
      .pivot(index='line', columns='year', values='bunching_rate')
      .round(2).to_string())

In [ ]:
# Annual total events (service volume proxy) — all years including 2026 partial
annual_vol = (
    agg.groupby(['year', 'line'])['n_events'].sum()
    .reset_index()
)

# Normalise 2026 to full-year estimate (128 days → 365 days equiv)
annual_vol_adj = annual_vol.copy()
annual_vol_adj.loc[annual_vol_adj['year'] == 2026, 'n_events'] *= (365 / 128)

fig, ax = plt.subplots(figsize=(13, 4))
for line in sorted(annual_vol['line'].unique()):
    sub = annual_vol_adj[annual_vol_adj['line'] == line].sort_values('year')
    ls  = '--' if 2026 in sub['year'].values else '-'
    ax.plot(sub['year'], sub['n_events'] / 1e6,
            color=LINE_COLORS.get(line, 'gray'), lw=1.8, marker='s', ms=5,
            label=line.replace(' Line', '').replace(' Trolley', ''))

ax.set_xlabel('Year')
ax.set_ylabel('Total Stop Events (millions)')
ax.set_title('Annual Service Volume by Line — 2022–2026 (2026 extrapolated)')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 4. Temporal Patterns — Hour of Day & Day of Week

In [ ]:
# Average bunching rate by hour (weekday vs weekend) — from aggregate_full
# Lines with headway data only
hourly_bunch = (
    agg[agg['line'].isin(hw_lines_agg)]
    .groupby(['hour', 'is_weekend'])
    .agg(bunching_events=('bunching_events', 'sum'),
         n_headway=('n_headway', 'sum'))
    .assign(bunching_rate=lambda d: d['bunching_events'] / d['n_headway'] * 100)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4))
for is_wknd, label, color, ls in [
    (False, 'Weekday',  'steelblue', '-'),
    (True,  'Weekend',  'darkorange', '--'),
]:
    sub = hourly_bunch[hourly_bunch['is_weekend'] == is_wknd].sort_values('hour')
    ax.plot(sub['hour'], sub['bunching_rate'], lw=2, color=color, ls=ls, label=label)

ax.axvspan(7, 9,   alpha=0.08, color='red',    label='AM Rush (7-9h)')
ax.axvspan(16, 18, alpha=0.08, color='orange', label='PM Rush (16-18h)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Bunching Rate (%)')
ax.set_title('Bunching Rate by Hour of Day — 2022–2026 average (Green + Red lines)')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Median dwell time by hour per line — weekdays only
# From aggregate_full: mean of station-level median_dwell, grouped by hour+line
dwell_hour = (
    agg[~agg['is_weekend']]
    .groupby(['line', 'hour'])
    .apply(lambda g: np.average(g['median_dwell'].dropna(),
                                 weights=g.loc[g['median_dwell'].notna(), 'n_events'])
           if g['median_dwell'].notna().any() else np.nan)
    .rename('wmean_dwell')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
for line in sorted(dwell_hour['line'].unique()):
    sub = dwell_hour[dwell_hour['line'] == line].sort_values('hour')
    ax.plot(sub['hour'], sub['wmean_dwell'],
            color=LINE_COLORS.get(line, 'gray'), lw=1.8,
            label=line.replace(' Line', '').replace(' Trolley', ''))

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Weighted Mean Dwell Time (sec)')
ax.set_title('Dwell Time by Hour (Weekdays) — 2022–2026 average')
ax.set_xticks(range(0, 24))
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 5. Initial Bunching Signal — Prevalence & Hotspots

In [ ]:
# Overall bunching rate by line — full 2022-2026 dataset
line_bunching = (
    agg.groupby('line')
    .agg(bunching_events=('bunching_events', 'sum'),
         n_headway=('n_headway', 'sum'))
    .assign(bunching_rate=lambda d: d['bunching_events'] / d['n_headway'] * 100)
    .dropna(subset=['bunching_rate'])
    .sort_values('bunching_rate', ascending=False)
)
line_bunching['bunching_rate'] = line_bunching['bunching_rate'].round(2)

print('Bunching rate (headway < 2 min) by line — full 2022–2026:')
print(line_bunching[['bunching_events', 'n_headway', 'bunching_rate']].to_string())

In [ ]:
# Top 20 stations by total bunching events (all years combined)
station_bunching = (
    agg.groupby(['parent_station', 'line'])
    .agg(bunching_events=('bunching_events', 'sum'),
         n_headway=('n_headway', 'sum'))
    .assign(bunching_rate=lambda d: (d['bunching_events'] / d['n_headway'] * 100).round(2))
    .reset_index()
    .sort_values('bunching_events', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [LINE_COLORS.get(r, 'gray') for r in station_bunching['line']]
bars   = ax.barh(station_bunching['parent_station'],
                 station_bunching['bunching_events'] / 1000,
                 color=colors, alpha=0.8)
ax.set_xlabel('Total Bunching Events (thousands, 2022–2026)')
ax.set_title('Top 20 Stations by Bunching Events — 2022–2026')
ax.invert_yaxis()
for bar, (_, row) in zip(bars, station_bunching.iterrows()):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['bunching_rate']:.1f}%  {row['line'].replace(' Line','').replace(' Trolley','')}",
            va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Bunching rate by hour — weekday vs weekend (all years)
hourly_lines = (
    agg[agg['line'].isin(hw_lines_agg) & ~agg['is_weekend']]
    .groupby('hour')
    .agg(bunching_events=('bunching_events', 'sum'),
         n_headway=('n_headway', 'sum'))
    .assign(bunching_rate=lambda d: d['bunching_events'] / d['n_headway'] * 100)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(hourly_lines['hour'], hourly_lines['bunching_rate'],
       color='steelblue', alpha=0.8)
ax.axvspan(7,  9,  alpha=0.12, color='red',    label='AM Rush (7-9h)')
ax.axvspan(16, 18, alpha=0.12, color='orange', label='PM Rush (16-18h)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Bunching Rate (%)')
ax.set_title('Bunching Rate by Hour — Weekdays, 2022–2026 average (Green + Red lines)')
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend()
plt.tight_layout()
plt.show()

## 6. Headway Gap → Dwell Time — Crowding Inference Preview (H2)

**Hypothesis H2:** Larger preceding headway gap → more accumulated passengers → longer dwell time.

Row-level analysis: load 3 strategic months (Jan, Apr, Jul 2024) representing winter, spring, summer.

In [ ]:
# Load 3 seasonal months for H2 row-level analysis
h2_frames = []
for ym in ['2024-01', '2024-04', '2024-07']:
    tmp = pd.read_parquet(
        PROC_DIR / 'strategic' / f'{ym}.parquet',
        columns=['route_id', 'line', 'parent_station',
                 'headway_branch_seconds', 'dwell_time_seconds', 'stop_timestamp']
    )
    tmp['month_label'] = ym
    h2_frames.append(tmp)

h2 = pd.concat(h2_frames, ignore_index=True)
h2 = h2[
    h2['headway_branch_seconds'].notna() &
    h2['dwell_time_seconds'].notna() &
    (h2['dwell_time_seconds'] > 0) &
    (h2['headway_branch_seconds'] > 0)
].copy()

print(f'H2 dataset: {len(h2):,} rows across 3 months')
print(f'Lines with headway data: {h2.groupby("line")["headway_branch_seconds"].count().sort_values(ascending=False).to_dict()}')

In [ ]:
# Binned median: headway gap vs. dwell time per line
hw_cap    = h2['headway_branch_seconds'].quantile(0.97)
dwell_cap = h2['dwell_time_seconds'].quantile(0.97)
plot_df   = h2[(h2['headway_branch_seconds'] <= hw_cap) &
               (h2['dwell_time_seconds'] <= dwell_cap)].copy()

plot_df['hw_bin'] = pd.cut(plot_df['headway_branch_seconds'],
                            bins=range(0, int(hw_cap)+60, 60), right=False)
binned = (
    plot_df.groupby(['line', 'hw_bin'])['dwell_time_seconds']
    .median().reset_index()
)
binned['hw_mid'] = binned['hw_bin'].apply(lambda x: x.mid)

lines_with_hw = (
    h2.groupby('line')['headway_branch_seconds'].count()
    .pipe(lambda s: s[s > 200].index.tolist())
)

fig, ax = plt.subplots(figsize=(13, 5))
for line in lines_with_hw:
    sub = binned[binned['line'] == line].dropna()
    ax.plot(sub['hw_mid'], sub['dwell_time_seconds'],
            color=LINE_COLORS.get(line, 'gray'), lw=2, marker='o', ms=4,
            label=line.replace(' Line', '').replace(' Trolley', ''))

ax.set_xlabel('Prior Headway Gap (sec) — binned by 60s')
ax.set_ylabel('Median Dwell Time (sec)')
ax.set_title('Prior Headway Gap vs. Dwell Time — Jan/Apr/Jul 2024 (3-month sample)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Global Pearson r and per-line
overall_r = h2['headway_branch_seconds'].corr(h2['dwell_time_seconds'])
print(f'\nGlobal Pearson r (headway → dwell): {overall_r:.4f}')
print()
per_line_r = (
    h2.groupby('line')
    .apply(lambda g: g['headway_branch_seconds'].corr(g['dwell_time_seconds']))
    .round(4)
    .rename('pearson_r')
    .sort_values(ascending=False)
)
print('Per-line Pearson r:')
print(per_line_r.to_string())
print()
print('Note: low global r expected — confounded by stop, time-of-day, line.')
print('Controlled regression (stop + hour fixed effects) in 03_bunching_analysis.ipynb.')

## 7. Green Line B Spotlight — Surface vs. Underground (H3)

**Hypothesis H3:** Surface stops (no fare gates) have shorter dwell times than underground stops.

Using all available strategic data — **2024-01 to 2026-05 (29 months)** — for maximum statistical power.

In [ ]:
SURFACE_STOPS = {
    'place-bland', 'place-brico', 'place-harvd', 'place-patk',
    'place-babck', 'place-plsgr', 'place-sthst', 'place-chswk',
    'place-sumav', 'place-grigg', 'place-alsgr', 'place-wrnst',
    'place-wascm', 'place-bc',
}

# Load all 29 months of strategic data (2024-01 → 2026-05), Green-B only
strategic_dir = PROC_DIR / 'strategic'
all_months = sorted(strategic_dir.glob('*.parquet'))   # 2024-01 through 2026-05

gb_frames = []
for f in all_months:
    tmp = pd.read_parquet(
        f,
        columns=['route_id', 'parent_station', 'stop_id',
                 'dwell_time_seconds', 'headway_branch_seconds', 'stop_timestamp',
                 'service_date']
    )
    gb_frames.append(tmp[tmp['route_id'] == 'Green-B'])

green_b = pd.concat(gb_frames, ignore_index=True)
green_b['is_surface'] = green_b['parent_station'].isin(SURFACE_STOPS)
green_b['stop_type']  = green_b['is_surface'].map({True: 'Surface', False: 'Underground'})
green_b['stop_dt']    = (pd.to_datetime(green_b['stop_timestamp'], unit='s', utc=True)
                           .dt.tz_convert('America/New_York'))
green_b['hour']       = green_b['stop_dt'].dt.hour
green_b['year']       = pd.to_datetime(green_b['service_date']).dt.year

print(f'Green Line B 2024–2026/05: {len(green_b):,} stop events across {green_b["year"].nunique()} years')
print(f'Surface stops : {green_b["is_surface"].sum():,}')
print(f'Underground   : {(~green_b["is_surface"]).sum():,}')
print()
print('Events by year:')
print(green_b.groupby('year').size().to_string())

In [ ]:
# Dwell time distribution: surface vs underground
dwell_cap = green_b['dwell_time_seconds'].quantile(0.97)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, stype, color in [
    (axes[0], 'Surface',     '#00843D'),
    (axes[1], 'Underground', '#555555'),
]:
    data = green_b.loc[
        (green_b['stop_type'] == stype) &
        green_b['dwell_time_seconds'].notna() &
        (green_b['dwell_time_seconds'] <= dwell_cap),
        'dwell_time_seconds'
    ]
    ax.hist(data, bins=60, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(data.median(), color='black', ls='--', lw=1.5,
               label=f'Median: {data.median():.0f}s')
    ax.axvline(data.mean(), color='red', ls=':', lw=1.5,
               label=f'Mean: {data.mean():.0f}s')
    ax.set_title(f'Green B — {stype} Stops')
    ax.set_xlabel('Dwell Time (sec)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

fig.suptitle('Green Line B: Dwell Time — Surface vs. Underground (2024–2026/05, 29 months)', fontsize=13)
plt.tight_layout()
plt.show()

# Summary stats
summary = (
    green_b[green_b['dwell_time_seconds'].notna()]
    .groupby('stop_type')['dwell_time_seconds']
    .agg(['median', 'mean', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75), 'count'])
    .rename(columns={'<lambda_0>': 'p25', '<lambda_1>': 'p75'})
    .round(1)
)
print('Dwell time summary — Green Line B 2024–2026/05:')
print(summary.to_string())

In [ ]:
# Per-stop dwell ranking — identify terminals and transfer hubs as outliers
stop_dwell = (
    green_b[green_b['dwell_time_seconds'].notna()]
    .groupby(['parent_station', 'stop_type'])
    .agg(
        median_dwell = ('dwell_time_seconds', 'median'),
        mean_dwell   = ('dwell_time_seconds', 'mean'),
        n            = ('dwell_time_seconds', 'count'),
    )
    .reset_index()
    .sort_values('median_dwell')
)

fig, ax = plt.subplots(figsize=(12, 8))
colors = stop_dwell['stop_type'].map({'Surface': '#00843D', 'Underground': '#555555'})
ax.barh(stop_dwell['parent_station'], stop_dwell['median_dwell'],
        color=colors, alpha=0.8)
ax.set_xlabel('Median Dwell Time (sec)')
ax.set_title('Green Line B: Median Dwell Time per Stop — 2024–2026/05 (29 months)')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#00843D', label='Surface'),
    Patch(color='#555555', label='Underground'),
], loc='lower right')
plt.tight_layout()
plt.show()

print(stop_dwell[['parent_station', 'stop_type', 'median_dwell', 'mean_dwell', 'n']].to_string(index=False))

## 8. Seasonality — Full 5-Year Monthly View (2022–2026)

In [ ]:
# Monthly bunching rate time series — all lines, 2022-2026
monthly_bunch = (
    agg[agg['line'].isin(hw_lines_agg)]
    .groupby(['year', 'month', 'line'])
    .agg(bunching_events=('bunching_events', 'sum'),
         n_headway=('n_headway', 'sum'))
    .assign(bunching_rate=lambda d: d['bunching_events'] / d['n_headway'] * 100)
    .reset_index()
)
monthly_bunch['period'] = (monthly_bunch['year'].astype(str) + '-' +
                            monthly_bunch['month'].astype(str).str.zfill(2))
monthly_bunch = monthly_bunch.sort_values('period')

fig, ax = plt.subplots(figsize=(16, 5))
for line in sorted(monthly_bunch['line'].unique()):
    sub = monthly_bunch[monthly_bunch['line'] == line]
    ax.plot(range(len(sub)), sub['bunching_rate'],
            color=LINE_COLORS.get(line, 'gray'), lw=1.5, alpha=0.9,
            label=line.replace(' Line', '').replace(' Trolley', ''))

# Year boundary lines
periods = monthly_bunch[monthly_bunch['line'] == monthly_bunch['line'].iloc[0]]['period'].tolist()
for yr in [2023, 2024, 2025, 2026]:
    try:
        idx = next(i for i, p in enumerate(periods) if p.startswith(str(yr)))
        ax.axvline(idx, color='gray', ls=':', lw=0.8, alpha=0.6)
        ax.text(idx + 0.3, ax.get_ylim()[1] * 0.95, str(yr), fontsize=8, color='gray')
    except StopIteration:
        pass

# X-axis: show year labels
tick_pos  = [i for i, p in enumerate(periods) if p.endswith('-01')]
tick_lbls = [p[:4] for p in periods if p.endswith('-01')]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_lbls)
ax.set_xlabel('Month')
ax.set_ylabel('Bunching Rate (%)')
ax.set_title('Monthly Bunching Rate — 2022–2026 (Green + Red lines)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly service volume — total stop events across all lines
monthly_vol = (
    agg.groupby(['year', 'month'])['n_events'].sum()
    .reset_index()
)
monthly_vol['period'] = (monthly_vol['year'].astype(str) + '-' +
                          monthly_vol['month'].astype(str).str.zfill(2))
monthly_vol = monthly_vol.sort_values('period')

fig, ax = plt.subplots(figsize=(16, 3))
ax.fill_between(range(len(monthly_vol)), monthly_vol['n_events'] / 1e6,
                alpha=0.6, color='steelblue')
ax.plot(range(len(monthly_vol)), monthly_vol['n_events'] / 1e6,
        color='steelblue', lw=1.2)

periods_v = monthly_vol['period'].tolist()
tick_pos  = [i for i, p in enumerate(periods_v) if p.endswith('-01')]
tick_lbls = [p[:4] for p in periods_v if p.endswith('-01')]
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_lbls)
ax.set_ylabel('Stop Events (M)')
ax.set_title('Monthly Service Volume — 2022–2026 (all lines)')
plt.tight_layout()
plt.show()

# Seasonal index: avg events per month-of-year
seasonal_index = (
    monthly_vol.groupby('month')['n_events'].mean()
    .rename('avg_events')
)
print('\nAverage monthly volume (seasonal index):')
print(seasonal_index.to_string())

## 9. Save Enriched Sample for Downstream Notebooks

In [ ]:
# Refresh sample_jan2024_enriched.parquet from strategic data (consistent with new pipeline)
def bunching_tier(hw):
    if pd.isna(hw):  return 'No data'
    if hw < 60:      return 'Severe   (<1 min)'
    if hw < 120:     return 'Moderate (1-2 min)'
    if hw < 240:     return 'Mild     (2-4 min)'
    return 'Normal'

def period_of_day(h):
    if 7 <= h < 9:   return 'AM Rush'
    if 9 <= h < 16:  return 'Midday'
    if 16 <= h < 19: return 'PM Rush'
    if 19 <= h < 23: return 'Evening'
    return 'Night/Early'

jan = pd.read_parquet(PROC_DIR / 'strategic' / '2024-01.parquet')
jan['stop_dt']      = (pd.to_datetime(jan['stop_timestamp'], unit='s', utc=True)
                        .dt.tz_convert('America/New_York'))
jan['hour']         = jan['stop_dt'].dt.hour
jan['dow']          = jan['stop_dt'].dt.dayofweek
jan['is_weekend']   = jan['dow'] >= 5
jan['is_bunched']   = jan['headway_branch_seconds'] < 120
jan['bunching_tier']= jan['headway_branch_seconds'].apply(bunching_tier)
jan['is_surface']   = jan['parent_station'].isin(SURFACE_STOPS)
jan['period']       = jan['hour'].apply(period_of_day)

out_path = PROC_DIR / 'sample_jan2024_enriched.parquet'
jan.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')
print(f'Shape: {jan.shape}')
print(f'New columns: {[c for c in jan.columns if c not in ["route_id","direction_id","trip_id","vehicle_id","stop_id","parent_station","stop_sequence","move_timestamp","stop_timestamp","travel_time_seconds","dwell_time_seconds","headway_branch_seconds","headway_trunk_seconds","scheduled_arrival_time","scheduled_departure_time","service_date","line"]]}'
      )

## Summary

### Data Upgrade vs. Previous Version
| Item | Before | After |
|------|--------|-------|
| Primary dataset | Jan 2024 only (885K rows) | aggregate_full 2022–2026 (4.4M rows) |
| Seasonality | 4 months fetched live | Full 5-year monthly view |
| Green B analysis | Jan 2024 (1 month) | **2024–2026/05 (29 months)** |
| H2 analysis | Jan 2024 only | Jan + Apr + Jul 2024 (3 seasons) |

### Bunching (from aggregate_full 2022–2026)
- Green lines dominate bunching; Blue/Orange/Mattapan have no branch headway data in LAMP.
- Year-over-year trend visible in Section 3 chart — check for post-COVID recovery pattern.
- Bunching peaks in specific hour windows; see Section 5 for hotspot stations.

### Headway → Dwell (H2 Preview)
- Global Pearson r is low (~0.05) — expected, heavy confounding by stop and time.
- Controlled OLS with stop + hour fixed effects in `03_bunching_analysis.ipynb`.

### Green Line B Surface vs. Underground (H3)
- Raw medians from 29 months of data show surface dwell ≈ underground dwell or slightly higher.
- H3 ("surface shorter due to no fare gates") not supported by raw dwell alone.
- Interpretation: surface stops have traffic light delays (friction), partially offsetting any fare-gate speedup.
- Formal Mann-Whitney test with corrected direction (`alternative='greater'`) in `03_bunching_analysis.ipynb`.

### Saved Outputs
- `data/processed/sample_jan2024_enriched.parquet` — Jan 2024 with engineered features for notebook 03

**Next:** `03_bunching_analysis.ipynb` — updated to use aggregate_full + strategic multi-month data.